In [ ]:
import numpy as np
import pandas as pd
import os
import re
import tensorflow as tf
from tqdm import tqdm
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.offline import init_notebook_mode
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Activation, Dropout, Flatten, Dense, Input, Layer
from tensorflow.keras.layers import BatchNormalization, Conv2DTranspose, LeakyReLU, Reshape, Activation
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

init_notebook_mode(connected=True)

# What are GANs ?

<div class='alert alert-success'><strong>Generative Adversarial Networks, or GANs for short, are an approach to generative modeling using deep learning methods, such as convolutional neural networks.</strong></div>

- Generative modeling is an unsupervised learning task in machine learning that involves automatically discovering and learning the regularities or patterns in input data in such a way that the model can be used to generate or output new examples that plausibly could have been drawn from the original dataset.
- GANs are a clever way of training a generative model by framing the problem as a supervised learning problem with two sub-models: the generator model that we train to generate new examples, and the discriminator model that tries to classify examples as either real (from the domain) or fake (generated). The two models are trained together in a zero-sum game, adversarial, until the discriminator model is fooled about half the time, meaning the generator model is generating plausible examples.
- GANs are an exciting and rapidly changing field, delivering on the promise of generative models in their ability to generate realistic examples across a range of problem domains, most notably in image-to-image translation tasks such as translating photos of summer to winter or day to night, and in generating photorealistic photos of objects, scenes, and people that even humans cannot tell are fake.

__Face Generation__
- In this notebook, the focus will be on generating new faces using some of the real facial images.
- The generator will try to mimic the features identified by the CNNs and produce some image
- The discriminator will try to classify if the generated image is fake or real
- Both these networks will be trained simultaneously

__Important Steps__
- Use a tanh activation in the final layer of the generator
- Use Leaky ReLU with a slope of 0.2
- Use BatchNormalization
- Use Adam optimizer with a LR of 0.0002 and Beta 1 of 0.5
- There are several more configurations stated in the original paper and how they were identified using finetuning. Link to the paper: https://arxiv.org/pdf/1511.06434.pdf

<img src='https://media.arxiv-vanity.com/render-output/6488257/INCFIGS/fig8.png'>

In [ ]:
img_size = 64
batch_size = 64

def scaling(x):
    x = (x-127.5)/127.5 
    return x
    

datagen = ImageDataGenerator(preprocessing_function=scaling)

data_generator = datagen.flow_from_directory(directory='/kaggle/input/celeba-face-recognition-triplets/CelebA FR Triplets',
                                        target_size=(img_size,img_size),
                                        class_mode=None,
                                        batch_size=batch_size,
                                        classes=['images'],
                                        shuffle=True)

samples = data_generator.samples

# Visualization of Images

In [ ]:
images = data_generator.__next__()

plt.figure(figsize = (20 , 20))
for i in range(15):
    plt.subplot(5 , 5, i+1)
    plt.subplots_adjust(hspace = 0.5 , wspace = 0.3)
    plt.imshow((images[i]+1)/2)
    plt.axis('off')

# DC-GANS Architecture

<div class='alert alert-info'><strong>Note:</strong> Both the generator and the discriminator are neural networks. The generator output is connected directly to the discriminator input. Through backpropagation, the discriminator's classification provides a signal that the generator uses to update its weights.</div>

## Discriminator
The discriminator has two sources of training data
- __Real Data:__ data of real objects, instances, people, etc. The discriminator treats these as positive examples during the training process.
- __Fake Data:__ data samples created by the generator. These are regarded as negative examples during the training process.

During the training process the discriminator ignore the generator loss and just uses the discriminator loss. The discriminator loss penalizes itself for misclassification of real instance as fake or otherwise, followed by which it updates the weights of the discriminator network through backpropagation. An illustration is given below. 

<img src='https://developers.google.com/static/machine-learning/gan/images/gan_diagram_discriminator.svg'>

In [ ]:
discriminator = Sequential(
    [
        Input(shape=(64, 64, 3)),
        Conv2D(32, kernel_size=5, strides=2, padding="same"),
        BatchNormalization(),
        LeakyReLU(alpha=0.2),
        Conv2D(64, kernel_size=5, strides=2, padding="same"),
        BatchNormalization(),
        LeakyReLU(0.2),
        Conv2D(128, kernel_size=5, strides=2, padding="same"),
        BatchNormalization(),
        LeakyReLU(0.2),
        Conv2D(256, kernel_size=5, strides=2, padding="same"),
        BatchNormalization(),
        LeakyReLU(0.2),
        Conv2D(256, kernel_size=5, strides=2, padding="same"),
        BatchNormalization(),
        LeakyReLU(0.2),
        Flatten(),
        Dropout(0.4),
        Dense(1, activation="sigmoid"),
    ],
    name="discriminator",
)
discriminator.summary()

## Generator

As stated earlier the generator learns to create fake data by incorporating the discriminator's feedback. The components of the generator's training process include
- random input (some latent space vector representation)
- generator network
- discriminator network and its output
- generator loss which penalizes the generator for failing to fool the discriminator

<div class='alert alert-info'><strong>Note:</strong> At the time of generator's training we do not want the discriminator to update its weights. Therefore we only use the discriminator here for its forward propagation output and do not update its weights. The backpropagation performed on the discriminator is used to update the weights of the generator.</div>

During the generator training process, we sample some random noise as input, perform a forward propagation using the generator network and get the discriminator to classify the generator output as __Fake__ or __Real__. Followed by this the loss of the discriminator's classification is calculated and generator's loss in terms of failure of fooling the discriminator is calculated. The gradients obtained through backpropagation of both the discriminator and generator are then used to update the weights of the generator. 

<img src='https://developers.google.com/static/machine-learning/gan/images/gan_diagram_generator.svg'>

In [ ]:
latent_dim = 256
generator = Sequential(
    [
        Input(shape=(latent_dim,)),
        Dense(4 * 4 * 256),
        Reshape((4, 4, 256)),
        Conv2DTranspose(256, kernel_size=4, strides=2, padding="same"),
        BatchNormalization(),
        LeakyReLU(alpha=0.2),
        Conv2DTranspose(128, kernel_size=4, strides=2, padding="same"),
        BatchNormalization(),
        LeakyReLU(alpha=0.2),
        Conv2DTranspose(64, kernel_size=4, strides=2, padding="same"),
        BatchNormalization(),
        LeakyReLU(alpha=0.2),
        Conv2DTranspose(64, kernel_size=4, strides=2, padding="same"),
        BatchNormalization(),
        LeakyReLU(alpha=0.2),
        Conv2D(3, kernel_size=5, padding="same", activation="tanh"),
    ],
    name="generator",
)
generator.summary()

# GAN Training !

- The training is done by alternating the forward-backward propagation of both the networks. The entire GAN training proceeds in alternating periods. 
- Here in this kernel, the discriminator trains for 1 epoch, followed by which the generator trains for 1 epoch.

The following loss function is used to train the discriminator

<img src='https://static.packt-cdn.com/products/9781789139907/graphics/bf03e5ab-69ac-424d-84a7-48ea85e616ec.png'>

The following loss function used to train the generator

<img src='https://static.packt-cdn.com/products/9781789139907/graphics/5d7ed5b7-0c10-4f09-bcbb-0585b65db52c.png'>


In [ ]:
gen_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
dis_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
loss_fn = tf.keras.losses.BinaryCrossentropy()
epochs=30
steps_per_epoch = int(-(samples/-batch_size))

disc_losses = []
gen_losses = []

for epoch in range(epochs+1):
    print("Epoch: ",epoch)
    for idx, (real) in enumerate(tqdm(data_generator)):
        if idx > steps_per_epoch:
            break
        
        batch_size = real.shape[0]
        random_latent_vectors = tf.random.normal(shape = (batch_size, latent_dim))
        fake = generator(random_latent_vectors)

        with tf.GradientTape() as d_tape:
            loss_disc_real = loss_fn(tf.ones((batch_size, 1)), discriminator(real))
            loss_disc_fake = loss_fn(tf.zeros((batch_size, 1)), discriminator(fake))
            loss_disc = (loss_disc_real + loss_disc_fake)/2
            disc_losses.append(loss_disc)

        grads = d_tape.gradient(loss_disc, discriminator.trainable_weights)
        dis_optimizer.apply_gradients(zip(grads, discriminator.trainable_weights))

        with tf.GradientTape() as g_tape:
            fake = generator(random_latent_vectors)
            output = discriminator(fake)
            loss_gen = loss_fn(tf.ones(batch_size, 1), output)
            gen_losses.append(loss_gen)

        grads = g_tape.gradient(loss_gen, generator.trainable_weights)
        gen_optimizer.apply_gradients(zip(grads, generator.trainable_weights))
    
    mean_disc_loss = sum(disc_losses)/len(disc_losses)
    mean_gen_loss = sum(gen_losses)/len(gen_losses)
    
    print(f"Epoch: {epoch} Generator Loss: {mean_gen_loss} Discriminator Loss: {mean_disc_loss}")
    data_generator.on_epoch_end()
    
    if epoch % 5 == 0:
        random_latent_vectors = tf.random.normal(shape = (10, latent_dim))
        fake = generator(random_latent_vectors)
        generated_images = fake.numpy()
        plt.figure(figsize = (20 , 6))
        for i in range(10):
            plt.subplot(2 , 5, i+1)
            plt.subplots_adjust(hspace = 0.5 , wspace = 0.3)
            image = generated_images[i]
            plt.imshow((image+1)/2)
            plt.axis('off')
        plt.show()

In [ ]:
random_latent_vectors = tf.random.normal(shape = (30, latent_dim))
fake = generator.predict(random_latent_vectors)

# Exploration of Latent Space
- Lets play around with the latent space features. We can identify the different features in the generated images like the gender of the face and whether the person is smiling or not
- Lets try to perform some latent space arithmatic to generate faces with specific attributes
- First we will index all the generated faces and assign them some labels

In [ ]:
plt.figure(figsize = (20 , 20))
for i in range(30):
    plt.subplot(6 , 5, i+1)
    plt.subplots_adjust(hspace = 0.5 , wspace = 0.3)
    plt.imshow((fake[i]+1)/2)
    plt.title(f"Index: {i}")
    plt.axis('off')
plt.show()

# Latent Space Arithmatic 
- Faces at index 18, 21 and 22 are of women who are smiling
- Faces at index 8, 17 and 29 are of women who have neutral expressions
- Faces at index 5, 9 and 10 are of men who have a neutral expression

Lets first create an average of the random latent vectors of these faces so that we get the robust latent space representations of these images. We then perform simple arithmetic operations on these latent vector representations.

Example:-

**Smiling Man** = **Smiling Woman** - **Neutral Woman** + **Neutral Man**

<img src='https://machinelearningmastery.com/wp-content/uploads/2019/05/Example-of-Vector-Arithmetic-on-Points-in-the-Latent-Space-for-Generating-Faces-with-a-GAN-1024x531.png'>

In [ ]:
smiling_woman = (random_latent_vectors[18] + random_latent_vectors[21] + random_latent_vectors[22]) / 3
neutral_woman = (random_latent_vectors[8] + random_latent_vectors[17] + random_latent_vectors[29]) / 3
neutral_man = (random_latent_vectors[5] + random_latent_vectors[9] + random_latent_vectors[10]) / 3

smiling_man = smiling_woman - neutral_woman + neutral_man

In [ ]:
sample1 = generator.predict(np.expand_dims(smiling_woman,axis=0))
sample2 = generator.predict(np.expand_dims(neutral_woman,axis=0))
sample3 = generator.predict(np.expand_dims(neutral_man,axis=0))
result = generator.predict(np.expand_dims(smiling_man,axis=0))

In [ ]:
plt.figure(figsize = (20 , 6))
plt.subplot(1,4,1)
plt.imshow((sample1[0]+1)/2)
plt.title("Smiling Woman")
plt.axis('off')
plt.subplot(1,4,2)
plt.imshow((sample2[0]+1)/2)
plt.title("Neutral Woman")
plt.axis('off')
plt.subplot(1,4,3)
plt.imshow((sample3[0]+1)/2)
plt.title("Neutral Man")
plt.axis('off')
plt.subplot(1,4,4)
plt.imshow((result[0]+1)/2)
plt.title("Smiling Man")
plt.axis('off')
plt.show()

# Conclusion
<div class='alert alert-success'>
<ul>
    <li>The <strong>hyperparameters for training GANs need to be selected very carefully</strong> as they can get unstable while training. Therefore the hyperparameters were selected in reference to the ones used in the original paper.</li>
<li>The faces that have been generated look pretty real and using arithmatic operations and interolation techniques we can further created new faces. We can also see that the <strong>generator network identifies the features of a human face very accurately</strong> to construct some facial images of its own.</li>
    <li>We were then able to utilize the learned features of the generator to perform <strong>Latent Space Arithmetics</strong> to generate faces with desired attributes</li>
    <li>The resolution of the generated images here is quite small (64x64x3), but we can progressively generate high resolution synthetic face images using a variant of GANs known as <strong>ProGANs or Progressive GANs</strong></li>
    </ul>

<strong>Sources</strong>
- https://developers.google.com/machine-learning/gan, https://en.wikipedia.org/wiki/Generative_adversarial_network
- https://arxiv.org/pdf/1511.06434.pdf
</div>